# Support vector Regressor

In [310]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

%matplotlib inline

from tqdm.auto import tqdm # used for progress bar

In [311]:
import warnings
warnings.filterwarnings('ignore')

In [312]:
dataset = sns.load_dataset('tips')
dataset

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [313]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [314]:
dataset['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [315]:
dataset['smoker'].value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [316]:
dataset['day'].value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

In [317]:
dataset['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [318]:
# filter out the independent and dependent feature 
# here target is bill

X = dataset[['sex','time','day','smoker','tip','size']]
y= dataset[['total_bill']]

In [319]:
X

,sex,time,day,smoker,tip,size
0,Female,Dinner,Sun,No,1.01,2
1,Male,Dinner,Sun,No,1.66,3
2,Male,Dinner,Sun,No,3.50,3
3,Male,Dinner,Sun,No,3.31,2
4,Female,Dinner,Sun,No,3.61,4
...,...,...,...,...,...,...
239,Male,Dinner,Sat,No,5.92,3
240,Female,Dinner,Sat,Yes,2.00,2
241,Male,Dinner,Sat,Yes,2.00,2
242,Male,Dinner,Sat,No,1.75,2


In [320]:
# train_test split before encoding to avoid the data leakage

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y, random_state=42,test_size=.25)


In [321]:
X_test['sex'].value_counts()

sex
Male      41
Female    20
Name: count, dtype: int64

In [322]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

le1 =LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

# fit_transform for  train dataset

X_train['sex'] = le1.fit_transform(X_train['sex'])
X_train['smoker'] = le2.fit_transform(X_train['smoker'])
X_train['time'] = le3.fit_transform(X_train['time'])

In [323]:
# transform only for test dataset

X_test['sex'] = le1.transform(X_test['sex'])
X_test['smoker'] = le2.transform(X_test['smoker'])
X_test['time'] = le3.transform(X_test['time'])

In [324]:
X_test

,sex,time,day,smoker,tip,size
24,1,0,Sat,0,3.18,2
6,1,0,Sun,0,2.00,2
153,1,0,Sun,0,2.00,4
211,1,0,Sat,1,5.16,4
198,0,1,Thur,1,2.00,2
...,...,...,...,...,...,...
172,1,0,Sun,1,5.15,2
242,1,0,Sat,0,1.75,2
152,1,0,Sun,0,2.74,3
231,1,0,Sat,1,3.00,3


In [325]:
## Onehot Encoding -- ColumnTransformer

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        (
        'onehot',
        OneHotEncoder(drop='first'),
       [2] # index on which we want to apply ...here day is at 2 
        # also we can use column names like instead of[2] using ['day']
        )
            ],

    remainder='passthrough' #without changing or dropping only pass
)


In [326]:
# ct.fit_transform(X_train)

In [327]:
import sys

np.set_printoptions(threshold=sys.maxsize)

X_train = ct.fit_transform(X_train)

In [328]:
X_test = ct.transform(X_test)

In [329]:
# SVR

from sklearn.svm import SVR

svr = SVR()

In [330]:
svr.fit(X_train,y_train)


SVR()

In [331]:
y_pred = svr.predict(X_test)

In [332]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

print(r2_score(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.49798620106004743
39.31122612339172
4.463296539661224


# Hypermeter Tuning

In [337]:
from sklearn.model_selection import GridSearchCV

gridCV = GridSearchCV(
     param_grid= {
        'C':[0.1,1.0,10],
        'gamma' : [1, 0.1, 0.01],
        'kernel':['rbf','poly','sigmoid','linear']
    },
    estimator=SVR(),
    scoring = 'r2',
    refit = True,
    cv = 5,
    n_jobs =-1,
    verbose = 3
)


In [338]:
gridCV.fit(X_train,y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


GridSearchCV(cv=5, estimator=SVR(), n_jobs=-1,
             param_grid={'C': [0.1, 1.0, 10], 'gamma': [1, 0.1, 0.01],
                         'kernel': ['rbf', 'poly', 'sigmoid', 'linear']},
             scoring='r2', verbose=3)

🧠 Why so many?

Your grid:

C → 3 values
gamma → 3 values
kernel → 4 values

👉 Total:

3×3×4=36 hyperparameter combination

Then:

36×5=180 fits  

In [339]:
gridCV.best_params_

{'C': 10, 'gamma': 1, 'kernel': 'linear'}

In [340]:
y_pred_grid = gridCV.predict(X_test)

In [341]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

print(r2_score(y_test,y_pred_grid))
print(mean_squared_error(y_test,y_pred_grid))
print(mean_absolute_error(y_test,y_pred_grid))

0.5921426223596538
31.938113319532526
4.1731609906016995
